# Sort by GSI annotation

Reads every JSON file in `fixed_batches_20_sorted_GSI/batch_XX/`, checks whether `fixed_text` contains a `[GSI = ...]` annotation, and copies the file into:
- `with_GSI/` — at least one `[GSI = ...]` found
- `without_GSI/` — no `[GSI = ...]` found

In [1]:
import json
import re
import shutil
from pathlib import Path

base_dir = Path("../fixed_batches_20_sorted_GSI")

gsi_pattern = re.compile(r'\[GSI\s*=', re.IGNORECASE)

stats = []

for batch_dir in sorted(base_dir.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith("batch_"):
        continue

    with_gsi_dir    = batch_dir / "with_GSI"
    without_gsi_dir = batch_dir / "without_GSI"
    with_gsi_dir.mkdir(exist_ok=True)
    without_gsi_dir.mkdir(exist_ok=True)

    n_with = n_without = 0

    for json_file in sorted(batch_dir.glob("*.json")):
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        fixed_text = data.get("fixed_text", "")

        if gsi_pattern.search(fixed_text):
            shutil.copy2(json_file, with_gsi_dir / json_file.name)
            n_with += 1
        else:
            shutil.copy2(json_file, without_gsi_dir / json_file.name)
            n_without += 1

    stats.append({"Batch": batch_dir.name, "with_GSI": n_with, "without_GSI": n_without,
                  "Total": n_with + n_without})
    print(f"{batch_dir.name}: {n_with} with GSI  |  {n_without} without GSI")

print("\nDone.")

batch_01: 26 with GSI  |  344 without GSI
batch_02: 25 with GSI  |  345 without GSI
batch_03: 22 with GSI  |  348 without GSI
batch_04: 49 with GSI  |  321 without GSI
batch_05: 22 with GSI  |  348 without GSI
batch_06: 33 with GSI  |  337 without GSI
batch_07: 25 with GSI  |  345 without GSI
batch_08: 20 with GSI  |  350 without GSI
batch_09: 25 with GSI  |  345 without GSI
batch_10: 35 with GSI  |  335 without GSI
batch_11: 32 with GSI  |  338 without GSI
batch_12: 22 with GSI  |  348 without GSI
batch_13: 16 with GSI  |  354 without GSI
batch_14: 30 with GSI  |  340 without GSI
batch_15: 30 with GSI  |  340 without GSI
batch_16: 15 with GSI  |  355 without GSI
batch_17: 34 with GSI  |  336 without GSI
batch_18: 25 with GSI  |  345 without GSI
batch_19: 27 with GSI  |  343 without GSI
batch_20: 7 with GSI  |  244 without GSI

Done.


In [2]:
import pandas as pd

df = pd.DataFrame(stats)

totals = df[["with_GSI", "without_GSI", "Total"]].sum().to_dict()
totals["Batch"] = "TOTAL"
df_display = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)

df_display

,Batch,with_GSI,without_GSI,Total
0,batch_01,26,344,370
1,batch_02,25,345,370
2,batch_03,22,348,370
3,batch_04,49,321,370
4,batch_05,22,348,370
5,batch_06,33,337,370
6,batch_07,25,345,370
7,batch_08,20,350,370
8,batch_09,25,345,370
9,batch_10,35,335,370


In [3]:
# Collect all with_GSI files from batch_06 to batch_20 into a single flat folder
all_gsi_dir = Path("..") / "all_GSI_files"
all_gsi_dir.mkdir(exist_ok=True)

copied = 0
for batch_dir in sorted(base_dir.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith("batch_"):
        continue
    batch_num = int(batch_dir.name.split("_")[1])
    if batch_num < 6:
        continue
    for json_file in sorted((batch_dir / "with_GSI").glob("*.json")):
        shutil.copy2(json_file, all_gsi_dir / json_file.name)
        copied += 1

print(f"Copied {copied} files to: {all_gsi_dir.resolve()}")

Copied 376 files to: /Users/somaya/Documents/Phd/github_version/version_2/deb-data-pipeline-github/data_verification/all_GSI_files


In [4]:
# Create a subfolder per species inside all_GSI_files and move each file into it
for json_file in list(all_gsi_dir.glob("*.json")):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    species_dir = all_gsi_dir / data["specie"]
    species_dir.mkdir(exist_ok=True)
    json_file.rename(species_dir / json_file.name)

n_species = sum(1 for d in all_gsi_dir.iterdir() if d.is_dir())
print(f"{n_species} species folders created in {all_gsi_dir.resolve()}")

376 species folders created in /Users/somaya/Documents/Phd/github_version/version_2/deb-data-pipeline-github/data_verification/all_GSI_files
